# Read RMF table

In [1]:
import pandas as pd
import numpy as np
RFM = pd.read_csv('RMF_table.csv')

In the following section, for the purpose of segmentation [Champions, Loyal, Big Spenders, At Risk, Lost], I give a score to each customer for each RMF feature based on quantile analysis. `RMF_score = 532` means the customer got `R_score = 5`, `M_score = 3`, and `Frequency_score = 2`. For the Frequency and Recency features, since their values are discrete, I ranked all data entries. Customers with identical values received the same average rank, ensuring that tied values are treated equally. Imagine that you have 100 data entries with `frequency=1`. If you use quantile-based scoring, you would give different `frequency_score` to different customers, although they have the same value. So that is why I came up with this approach of ranking.

In [2]:
# RFM['R_score'] = pd.qcut(RFM['Recency_Days'], 5, labels=[5,4,3,2,1])
rec_pct = RFM['Recency_Days'].rank(method='average', pct=True)
RFM['R_score'] = (6 - np.ceil(rec_pct*5)).astype(int).clip(1,5)


# RFM['Frequency_score'] = pd.qcut(RFM['Frequency'].rank(method = 'first'), 5, labels=[1,2,3,4,5])
freq_pct = RFM['Frequency'].rank(method = 'average', pct = True)
RFM['F_score'] = np.ceil(freq_pct * 5).astype(int).clip(1,5)

RFM['M_score'] = pd.qcut(RFM['Monetary'], 5, labels=[1,2,3,4,5])

RFM['RFM_score'] = RFM['R_score'].astype(str) + RFM['F_score'].astype(str) + RFM['M_score'].astype(str)
display(RFM.head())

,CustomerID,Recency_Days,Frequency,Monetary,R_score,F_score,M_score,RFM_score
0,12346.0,325.0,12,77556.46,2,5,5,255
1,12347.0,2.0,8,5633.32,5,4,5,545
2,12348.0,75.0,5,2019.40,3,4,4,344
3,12349.0,18.0,4,4428.69,5,3,5,535
4,12350.0,310.0,1,334.40,2,1,2,212


Now we can segment customers based on the RFM_score.

In [3]:
def segment_customer(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Champions'
    elif row['F_score'] >= 4 and row['R_score'] >= 3:
        return 'Loyal'
    elif row['M_score'] >= 4:
        return 'Big Spenders'
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return 'Lost'
    else:
        return 'Need Attention'

RFM['Segment'] = RFM.apply(segment_customer, axis=1)
display(RFM.head(50))

,CustomerID,Recency_Days,Frequency,Monetary,R_score,F_score,M_score,RFM_score,Segment
0,12346.0,325.0,12,77556.46,2,5,5,255,Big Spenders
1,12347.0,2.0,8,5633.32,5,4,5,545,Champions
2,12348.0,75.0,5,2019.40,3,4,4,344,Loyal
3,12349.0,18.0,4,4428.69,5,3,5,535,Big Spenders
4,12350.0,310.0,1,334.40,2,1,2,212,Lost
5,12351.0,375.0,1,300.93,2,1,2,212,Lost
6,12352.0,36.0,10,2849.84,4,5,4,454,Champions
7,12353.0,204.0,2,406.76,2,2,2,222,Lost
8,12354.0,232.0,1,1079.40,2,1,3,213,Lost
9,12355.0,214.0,2,947.61,2,2,3,223,Lost


Now let us visualize the segmented groups:

In [4]:
segment_summary = RFM.groupby('Segment').agg({
    'CustomerID': 'count',
    'Recency_Days': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).rename(columns={'CustomerID':'Count'}).sort_values('Count', ascending=False)

display(segment_summary)

,Count,Recency_Days,Frequency,Monetary
Segment,,,,
Lost,1571,457.184596,1.289624,341.842400
Need Attention,1358,61.231959,2.152430,537.796673
Champions,1271,19.151849,17.361133,9490.649477
Big Spenders,689,235.303338,4.946299,3347.408222
Loyal,606,85.722772,8.349835,2988.394546
At Risk,386,368.484456,3.865285,767.150363


In [12]:
segment_summary.to_csv("Segmented_customers.csv", index=True)
segment_summary.to_excel("Segmented_customers.xlsx", index=True)